# Spatial perturbation benchmark — LOCAL synthetic run
Runs the full 2x2 (seed x propagation) benchmark on **synthetic data** — no server, no real .h5mu, no extra deps (pandas not required). Use this to develop/inspect the pipeline locally.
The server version (`run_benchmark.ipynb`) is identical except its data source is the Saunders `.h5mu` adapter instead of the synthetic generator.

In [ ]:
%matplotlib inline
from spbench.synthetic import make_synthetic
from spbench.config import run_benchmark
from spbench.viz import (plot_2x2, plot_aggregate_2x2, plot_baseline_gain,
                         plot_gain_per_perturbation, plot_significance_contrast,
                         plot_learned_value)

## 1. Generate synthetic data (planted seed + propagation effects)

In [ ]:
data = make_synthetic(seed=0)
print('cells', data.n_cells, '| genes', data.n_genes,
      '| perturbations', data.perturbations())

## 2. Run the benchmark (trivial seed + Gaussian baseline + simple GCN)

In [ ]:
PERTS = data.perturbations()
res = run_benchmark(data, perturbations=PERTS, k=15, k_ref=5,
                    gcn_kwargs={'hidden': 32, 'epochs': 20})

## 3. Metric table (per perturbation): seed + propagation, decoupled
**seed**: `seed_pcc` (direction of the cell's own shift, higher better), `seed_mse` (magnitude). **propagation (deployable model+learned)**: `niche_gain = e_null − e` (>0 beats 'no effect'), `niche_pcc` (direction of the niche shift). `e_null` = baseline.

In [ ]:
hdr = f"{'pert':6} {'seed_pcc':>8} {'seed_mse':>8} | {'niche_gain':>10} {'niche_pcc':>9} {'e_null':>7}"
print(hdr); print('-' * len(hdr))
for p in PERTS:
    c = res['compare'][p]; s = res['seed'][p]
    print(f"{p:6} {s['pcc_delta']:8.3f} {s['mse']:8.3f} | "
          f"{c['gain']['model+learned']:10.3f} {c['pcc']['model+learned']:9.3f} {c['e']['null']:7.3f}")

## 4. Result figures
On synthetic data the seed is planted on gene i and the propagation on gene i+10, so models that 'spread the seed' predict the wrong gene — `gain` is correctly **negative** (worse than no effect) while the `oracle` ceiling is positive (there IS recoverable signal). That is the intended sanity check, not a bug. We mark a subset as significant just to demonstrate the colours; on real data pass the MC-spatial significant list.

In [ ]:
SIGNIFICANT = PERTS[:2]   # demo only; on real data use the MC-spatial significant list
plot_baseline_gain(res)                        # headline: each method vs the baseline

In [ ]:
plot_aggregate_2x2(res)                        # summary 2x2: mean gain over baseline

In [ ]:
plot_gain_per_perturbation(res, SIGNIFICANT)   # per gene: deployable model vs baseline

In [ ]:
plot_significance_contrast(res, SIGNIFICANT)   # is the GCN-vs-Gaussian edge signal-specific?

In [ ]:
plot_learned_value(res, SIGNIFICANT)           # learned_value (e1-e2) per perturbation

**Single-gene 2x2 (explainer, not a result):**

In [ ]:
plot_2x2(res['grids'][PERTS[0]], title=PERTS[0])

## 5. Bottom line (how many beat the no-effect baseline)

In [ ]:
g = {p: res['compare'][p]['gain']['model+learned'] for p in PERTS}
print('beat baseline (gain>0):', [p for p in PERTS if g[p] > 0])
print('ranking by end-to-end E-distance:', res['ranking'])